# 🚀 Marvedge — Task-00077: GPU Throughput & Cost Benchmark

**What this does:**  
Runs the *real* Marvedge ML pipeline (AVS face detection + reframe crop) on sample video clips with a live GPU VRAM / utilisation monitor, then produces a cost-per-minute table for Cloud Run L4 pricing.

> **Before you start:** Go to **Runtime → Change runtime type → Hardware accelerator → GPU (T4)**

In [ ]:
# ── Cell 1: Verify GPU ───────────────────────────────────────────────────────
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# ── Cell 2: Install deps ─────────────────────────────────────────────────────
!pip install -q opencv-python-headless scipy tqdm
# TalkNet-ASD provides the S3FD face detector
!git clone -q https://github.com/TaoRuijie/TalkNet-ASD.git /content/TalkNet-ASD
!pip install -q gdown
# Download S3FD weights
!gdown 1KafnHz7ccT-3IyddBsL5yi2xGtxAKypt -O /content/TalkNet-ASD/model/faceDetector/s3fd/sfd_face.pth
# Fix numpy 2.x compat
!sed -i 's/np\.int)/int)/g' /content/TalkNet-ASD/model/faceDetector/s3fd/box_utils.py
!sed -i 's/np\.float)/float)/g' /content/TalkNet-ASD/model/faceDetector/s3fd/box_utils.py
print('Done.')

In [ ]:
# ── Cell 3: Clone Marvedge + copy our benchmark script ───────────────────────
# Change REPO_URL to your actual GitHub URL if private
REPO_URL = 'https://github.com/Marvedge/marvedge.git'
BRANCH   = 'feat/task-77-gpu-benchmarking'

!git clone -q --branch {BRANCH} {REPO_URL} /content/marvedge
!cp /content/marvedge/ml-worker/preprocess_faces.py /content/TalkNet-ASD/preprocess_faces.py
print('Marvedge cloned.')

In [ ]:
# ── Cell 4: Upload your own clips ────────────────────────────────────────────
import os, subprocess, shutil
from google.colab import files

os.makedirs('/content/clips', exist_ok=True)
print('Please upload your video file (mp4, mov, etc.):')
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    dest = '/content/clips/30s.mp4'
    shutil.move(filename, dest)
    dur = subprocess.check_output(
        ['ffprobe','-v','error','-select_streams','v:0',
         '-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1', dest]
    ).decode().strip()
    print(f'
✅ Uploaded and prepared: {dest} ({float(dur):.1f}s)')
else:
    print('No file uploaded.')

In [ ]:
# ── Cell 5: The Benchmark Harness ────────────────────────────────────────────
import subprocess, threading, time, json, os

T4_COST_PER_SEC = 0.35 / 3600   # ~$0.000097/s  (Colab Pro+ T4 approx)
L4_COST_PER_SEC = 0.80 / 3600   # ~$0.000222/s  (Cloud Run L4)

def poll_gpu(stop_evt, samples, interval=0.1):
    cmd = ['nvidia-smi',
           '--query-gpu=memory.used,memory.total,utilization.gpu,temperature.gpu',
           '--format=csv,noheader,nounits']
    while not stop_evt.is_set():
        try:
            raw = subprocess.check_output(cmd, stderr=subprocess.DEVNULL, timeout=2)
            parts = raw.decode().strip().split(',')
            if len(parts) >= 4:
                samples.append({
                    'mem_used':  int(parts[0]),
                    'mem_total': int(parts[1]),
                    'util':      int(parts[2]),
                    'temp':      int(parts[3]),
                })
        except Exception:
            pass
        stop_evt.wait(interval)


def run_benchmark(stage, cmd, video_sec=None, gpu_type='T4'):
    cost_per_sec = T4_COST_PER_SEC if gpu_type == 'T4' else L4_COST_PER_SEC
    samples = []
    stop_evt = threading.Event()
    poller = threading.Thread(target=poll_gpu, args=(stop_evt, samples, 0.1), daemon=True)
    poller.start()

    t0 = time.perf_counter()
    proc = subprocess.run(cmd, shell=True, capture_output=False)
    t1 = time.perf_counter()

    stop_evt.set()
    poller.join()

    duration = t1 - t0
    cost     = duration * cost_per_sec
    max_mem  = max((s['mem_used']  for s in samples), default=0)
    mem_tot  = samples[0]['mem_total'] if samples else 0
    max_util = max((s['util']      for s in samples), default=0)
    avg_util = (sum(s['util'] for s in samples) / len(samples)) if samples else 0
    max_temp = max((s['temp']      for s in samples), default=0)

    result = {
        'stage': stage, 'gpu_type': gpu_type,
        'success': proc.returncode == 0,
        'duration_s': round(duration, 2),
        'video_s': video_sec,
        'realtime_factor': round(video_sec / duration, 2) if video_sec else None,
        'max_vram_mib': max_mem,
        'vram_total_mib': mem_tot,
        'vram_pct': round(max_mem/mem_tot*100, 1) if mem_tot else 0,
        'max_gpu_util': max_util,
        'avg_gpu_util': round(avg_util, 1),
        'max_temp_c': max_temp,
        'cost_usd': round(cost, 6),
        'cost_per_min_usd': round(cost_per_sec * 60, 4),
        'poll_samples': len(samples),
    }

    print(f"\n{'─'*55}")
    print(f"  {stage.upper():<12} | {'✅ PASS' if result['success'] else '❌ FAIL'}")
    print(f"  Duration      : {duration:.2f}s")
    if result['realtime_factor']:
        print(f"  Real-time     : {result['realtime_factor']}x")
    print(f"  Peak VRAM     : {max_mem} / {mem_tot} MiB  ({result['vram_pct']}%)")
    print(f"  GPU util      : peak {max_util}%  avg {avg_util:.1f}%  max_temp {max_temp}°C")
    print(f"  Cost ({gpu_type})   : ${cost:.6f}  (${cost_per_sec*60:.4f}/min)")
    print(f"{'─'*55}")
    return result

print('Harness loaded ✅')

In [ ]:
# ── Cell 6: Run AVS Stage (real S3FD face detection) ─────────────────────────
import os, subprocess

CLIP   = '/content/clips/30s.mp4'
OUTDIR = '/tmp/bench_avs_30s'
os.makedirs(OUTDIR, exist_ok=True)

# Get actual video duration
dur = float(subprocess.check_output(
    ['ffprobe','-v','error','-select_streams','v:0',
     '-show_entries','format=duration',
     '-of','default=noprint_wrappers=1:nokey=1', CLIP]
).decode().strip())

avs_cmd = (
    f'cd /content/TalkNet-ASD && '
    f'python3 preprocess_faces.py '
    f'  --videoPath {CLIP} '
    f'  --savePath  {OUTDIR} '
    f'  --nDataLoaderThread 4 '
    f'  --batchSize 32 '
    f'  --chunkSize 1000 '
    f'  --minTrack 5'
)

avs_result = run_benchmark('avs', avs_cmd, video_sec=dur, gpu_type='T4')

In [ ]:
# ── Cell 7: Run Reframe Stage (FFmpeg 9:16 crop) ─────────────────────────────
CLIP   = '/content/clips/30s.mp4'
OUTDIR = '/tmp/bench_reframe_30s'
os.makedirs(OUTDIR, exist_ok=True)

reframe_cmd = (
    f'ffmpeg -y -i {CLIP} '
    f'-vf "crop=in_h*9/16:in_h:(in_w-in_h*9/16)/2:0" '
    f'-c:v libx264 -preset fast -crf 22 '
    f'{OUTDIR}/reframe_out.mp4 -loglevel warning'
)

reframe_result = run_benchmark('reframe', reframe_cmd, video_sec=dur, gpu_type='T4')

In [ ]:
# ── Cell 8: Aggregate Results Table ──────────────────────────────────────────
import pandas as pd

all_results = [avs_result, reframe_result]

df = pd.DataFrame(all_results)[[
    'stage','video_s','duration_s','realtime_factor',
    'max_vram_mib','vram_pct','avg_gpu_util','max_temp_c','cost_usd'
]]
df.columns = [
    'Stage','Video(s)','Wall-clock(s)','x-Realtime',
    'Peak VRAM (MiB)','VRAM%','Avg GPU%','Temp(°C)','Cost(USD)'
]

print('\n=== MARVEDGE GPU BENCHMARK RESULTS ===')
print(df.to_string(index=False))

# Save to JSON for BENCHMARK_RESULTS.md
os.makedirs('/content/benchmark_results', exist_ok=True)
for r in all_results:
    with open(f"/content/benchmark_results/{r['stage']}_30s.json", 'w') as f:
        json.dump(r, f, indent=2)
print('\nReports saved to /content/benchmark_results/')

In [ ]:
# ── Cell 9: Extrapolate costs to L4 Cloud Run pricing ────────────────────────
L4 = 0.80 / 3600
print('=== CLOUD RUN L4 COST PROJECTION ===')
print(f'  (T4 → L4 is ~2.3x faster, so L4 realtime factor is higher)')
print()
for r in all_results:
    stage = r['stage']
    # L4 is roughly 2.3x faster than T4 for CNN inference
    l4_duration = r['duration_s'] / 2.3
    l4_cost     = l4_duration * L4
    l4_xrt      = r['video_s'] / l4_duration if r['video_s'] else None
    print(f"  {stage.upper():<12} | L4 est duration: {l4_duration:.2f}s | "
          f"cost: ${l4_cost:.6f} | realtime: {l4_xrt:.1f}x")